### Data Collection For Different Classes 

##### In this notebook, we will generate videos for different sign language classes. Each video will capture a person performing a specific sign gesture that will be used for training our sign language recognition model.


#### Use Python 3.11.9

In [2]:
!pip install "tensorflow>=2.16,<3" tensorflow-metal \
            "mediapipe>=0.10,<0.11" "opencv-python>=4.10,<5" \
            "numpy>=2.0,<2.3" "scikit-learn>=1.4,<1.6" "scipy>=1.11" matplotlib --quiet



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install --upgrade mediapipe opencv-python numpy matplotlib --quiet



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## 1. Import and Install Dependencies

In [4]:
import cv2, numpy as np
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

## 2. Keypoints using MP Holistic

In [5]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [6]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB - for mediapipe
    image.flags.writeable = False                  # Image is no longer writeable - saves a bit of memory
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR - for opencv
    return image, results

In [7]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw right hand connections

In [8]:
def draw_styled_landmarks(image, results):
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

## 3. Set Up Folders 

In [9]:
# Actions that we try to detect
actions = np.array(['MBS', 'Bedok', 'Clementi', 'Orchard', 'Esplanade', 'City Hall'])
# actions = np.array(['1', '2'])

# Thirty videos worth of data
no_sequences = 15

# Videos are going to be 30 frames in length
sequence_length = 60

# Folder start
start_folder = 0  # or compute from existing dirs if resuming

## 4. Record Videos

In [ ]:
# Directory to save short video clips per action/sequence
VIDEO_DATA_PATH = os.path.join('MP_Videos')
os.makedirs(VIDEO_DATA_PATH, exist_ok=True)

# Open webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError('Could not open webcam. Check camera permissions / device index.')

# Determine frame properties
frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480
fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 0:
    fps = 20.0  # sensible default if camera doesn't report FPS

should_exit = False

try:
    for action in actions:
        if should_exit:
            break
        action_dir = os.path.join(VIDEO_DATA_PATH, action)
        os.makedirs(action_dir, exist_ok=True)

        # Show action title screen (preview only)
        title_frame = np.zeros((frame_height, frame_width, 3), dtype=np.uint8)
        title_text = str(action)
        font = cv2.FONT_HERSHEY_SIMPLEX
        title_scale = 3.0
        title_thickness = 7
        text_size, _ = cv2.getTextSize(title_text, font, title_scale, title_thickness)
        text_x = (frame_width - text_size[0]) // 2
        text_y = (frame_height + text_size[1]) // 2
        cv2.putText(title_frame, title_text, (text_x, text_y), font, title_scale, (255, 255, 255), title_thickness, cv2.LINE_AA)
        cv2.imshow('OpenCV Feed', title_frame)
        if cv2.waitKey(1500) & 0xFF == ord('q'):
            should_exit = True

        # Loop through sequences (videos) for this action
        for sequence in range(start_folder, start_folder + no_sequences):
            if should_exit:
                break
            # Prepare video writer
            video_path = os.path.join(action_dir, f"{sequence}.mp4")
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(video_path, fourcc, fps, (frame_width, frame_height))

            # Fallback to AVI if MP4 encoder not available
            if not writer.isOpened():
                video_path = os.path.join(action_dir, f"{sequence}.avi")
                fourcc = cv2.VideoWriter_fourcc(*'MJPG')
                writer = cv2.VideoWriter(video_path, fourcc, fps, (frame_width, frame_height))

            if not writer.isOpened():
                print(f"[WARN] Could not open writer for {video_path}. Skipping sequence.")
                continue

            # 3-2-1 countdown before recording (preview only, not saved)
            for countdown in range(3, 0, -1):
                ok, frame_preview = cap.read()
                if not ok:
                    print("[WARN] Camera read failed during countdown; skipping sequence.")
                    break
                display = frame_preview.copy()
                label = str(countdown)
                font = cv2.FONT_HERSHEY_SIMPLEX
                scale = 3.0
                thickness = 6
                color = (0, 0, 255)
                text_size, _ = cv2.getTextSize(label, font, scale, thickness)
                text_x = (frame_width - text_size[0]) // 2
                text_y = (frame_height + text_size[1]) // 2
                cv2.putText(display, label, (text_x, text_y), font, scale, color, thickness, cv2.LINE_AA)
                cv2.putText(display, f'Preparing {action} Seq {sequence}', (15, 60),
                            font, 1.2, (255, 255, 255), 3, cv2.LINE_AA)
                cv2.imshow('OpenCV Feed', display)
                if cv2.waitKey(1000) & 0xFF == ord('q'):
                    should_exit = True
                    break

            if should_exit:
                writer.release()
                break

            # Record exactly `sequence_length` frames per sequence
            for frame_num in range(sequence_length):
                ok, frame = cap.read()
                if not ok:
                    print("[WARN] Camera read failed; stopping early for this sequence.")
                    break

                # Show status overlays on a preview copy only
                display_frame = frame.copy()
                if frame_num == 0:
                    cv2.putText(display_frame, 'STARTING COLLECTION', (80, 220),
                                cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 255, 0), 6, cv2.LINE_AA)
                    cv2.putText(display_frame, f'Collecting video for {action} Seq {sequence}', (15, 60),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3, cv2.LINE_AA)
                    cv2.imshow('OpenCV Feed', display_frame)
                    cv2.waitKey(500)
                else:
                    cv2.putText(display_frame, f'Collecting video for {action} Seq {sequence}', (15, 60),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3, cv2.LINE_AA)
                    cv2.imshow('OpenCV Feed', display_frame)

                # Write raw frame to video
                writer.write(frame)

                # Early exit
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    should_exit = True
                    break

            writer.release()

            if should_exit:
                break

finally:
    # Graceful closing screen
    closing_frame = np.zeros((frame_height, frame_width, 3), dtype=np.uint8)
    closing_text = 'Recording stopped' if should_exit else 'All recordings complete'
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 1.5
    thickness = 4
    text_size, _ = cv2.getTextSize(closing_text, font, scale, thickness)
    text_x = (frame_width - text_size[0]) // 2
    text_y = (frame_height + text_size[1]) // 2
    cv2.putText(closing_frame, closing_text, (text_x, text_y), font, scale, (255, 255, 255), thickness, cv2.LINE_AA)
    cv2.putText(closing_frame, 'Closing camera...', (text_x, min(frame_height - 20, text_y + 50)), font, 0.9, (200, 200, 200), 2, cv2.LINE_AA)
    cv2.imshow('OpenCV Feed', closing_frame)
    cv2.waitKey(1200)

    if cap is not None and cap.isOpened():
        cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)


: 